## Import Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, recall_score, accuracy_score, f1_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn import set_config

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import optuna
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
set_config(transform_output='pandas')

print('All core packages imported successfully.')


In [ ]:
try:
    import dagshub
    dagshub.init(repo_owner='Ankitkumar1141', repo_name='Churn-modelling', mlflow=True)
    print('DagsHub initialized successfully.')
except Exception as e:
    print(f'[DagsHub Notice] {e}')


In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri('https://dagshub.com/Ankitkumar1141/Churn-modelling.mlflow')
    mlflow.set_experiment('Experiment 4 - Model Selection')
    print('MLflow tracking active on DagsHub.')
except Exception as e:
    print(f'[MLflow Notice] {e}')


## Load the Preprocessed Data

In [ ]:
df = pd.read_csv('preprocessed.csv')
print('Loaded dataset shape:', df.shape)
df.head(3)


## Exploratory Checks

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False).head(5)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
print(df['churned'].value_counts())
churn_rate = df['churned'].mean()
print('\nChurn rate: {:.2%}'.format(churn_rate))


## Drop Unnecessary Columns

In [ ]:
drop_columns = [
    'has_balance',
    'products_per_tenure',
    'has_credit_card',
    'geo_france', 'geo_uk', 'geo_usa', 'geo_spain', 'geo_germany',
    'tenure', 'salary',
    'customer_id',
    'account_id', 'name', 'location_id',
    'age_when_joined', 'age',
    'churn_label',
    'has_credit_card_label',
    'is_active_label',
]

df = df.drop(columns=drop_columns)
print('Shape after dropping columns:', df.shape)
df.head(3)


In [ ]:
missing_cols = df.isna().any(axis=0).loc[lambda x: x].index.tolist()
print('Columns with missing values:', missing_cols)
pct_missing_rows = df.isna().any(axis=1).mean().round(4) * 100
print(f'Rows with missing values: {pct_missing_rows:.2f}%  ({df.isna().any(axis=1).sum():,} rows)')


## Train / Test Split

In [ ]:
X = df.drop(columns='churned')
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('Train size : {:,}  |  churn rate: {:.2%}'.format(X_train.shape[0], y_train.mean()))
print('Test  size : {:,}  |  churn rate: {:.2%}'.format(X_test.shape[0], y_test.mean()))


In [ ]:
print('Train shape:', X_train.shape)
print('Test  shape:', X_test.shape)
print('Missing in train (balance only):\n', X_train.isna().sum()[X_train.isna().sum() > 0])


## Column Type Identification

In [ ]:
numeric_cols     = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
binary_cols      = [c for c in numeric_cols if X[c].nunique() == 2]
scale_cols       = [c for c in numeric_cols if c not in binary_cols]

print('Numeric (to scale) :', scale_cols)
print('Binary pass-thru   :', binary_cols)
print('Categorical OHE    :', categorical_cols)


## Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scale',          MinMaxScaler(), scale_cols),
        ('nominal_encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform='pandas')

knn_imputer = KNNImputer(n_neighbors=10)

# Sanity check -- fit on train, transform test using flat steps
base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn_imputer',  knn_imputer),
])
X_train_trans = base_pipeline.fit_transform(X_train)
X_test_trans  = base_pipeline.transform(X_test)

print('X_train transformed shape:', X_train_trans.shape)
print('X_test  transformed shape:', X_test_trans.shape)


## Class Imbalance Handling — Winner Applied

From **Experiment 3 (class_imbalance.ipynb)**, `Random UnderSampling` achieved the highest
**CV Recall of 0.7110 ± 0.0355** and **Test Recall of 74.94%** across all 8 strategies tested.

This is the winning strategy applied to all model candidates in this spot-check.

| Winning Strategy | CV Recall | Test Recall | Test ROC-AUC |
|---|---|---|---|
| **Random UnderSample** | **0.7110 ± 0.0355** | **0.7494** | **0.8265** |


In [ ]:
# Winning imbalance strategy from Experiment 3
sampler = RandomUnderSampler(random_state=42)

# Apply resampling to training data only (never to test set)
X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_trans, y_train)

print('Before resampling — Train shape:', X_train_trans.shape)
print('Before resampling — Class distribution:')
print(y_train.value_counts())
print()
print('After  resampling — Train shape:', X_train_resampled.shape)
print('After  resampling — Class distribution:')
print(y_train_resampled.value_counts())


## Model Selection with Optuna

Multi-model spot-check over **7 model families** using **Optuna Bayesian optimisation**.  
Each trial is logged as a nested MLflow run.  
**Primary metric: ROC-AUC** (maximised).  

| Model | Key Hyperparameters |
|---|---|
| LR | `C` |
| DT | `max_depth` |
| KNN | `n_neighbors` |
| RF | `n_estimators`, `max_depth` |
| XGB | `n_estimators`, `learning_rate`, `max_depth` |
| LGBM | `n_estimators`, `learning_rate`, `max_depth` |
| SVM | `C` |


In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical(
            'model', ['LR', 'DT', 'KNN', 'RF', 'XGB', 'LGBM', 'SVM']
        )

        if model_name == 'LR':
            C_lr = trial.suggest_float('C_lr', 1e-3, 10.0, log=True)
            model = LogisticRegression(C=C_lr, max_iter=500, random_state=42)

        elif model_name == 'DT':
            max_depth_dt = trial.suggest_int('max_depth_dt', 3, 20)
            model = DecisionTreeClassifier(max_depth=max_depth_dt, random_state=42)

        elif model_name == 'KNN':
            n_neighbors_knn = trial.suggest_int('n_neighbors_knn', 3, 15)
            model = KNeighborsClassifier(n_neighbors=n_neighbors_knn, n_jobs=-1)

        elif model_name == 'RF':
            n_estimators_rf = trial.suggest_int('n_estimators_rf', 10, 200)
            max_depth_rf    = trial.suggest_int('max_depth_rf', 2, 20)
            model = RandomForestClassifier(
                n_estimators=n_estimators_rf,
                max_depth=max_depth_rf,
                random_state=42,
                n_jobs=-1,
            )

        elif model_name == 'XGB':
            n_estimators_xgb    = trial.suggest_int('n_estimators_xgb', 10, 200)
            learning_rate_xgb   = trial.suggest_float('learning_rate_xgb', 0.01, 0.5)
            max_depth_xgb       = trial.suggest_int('max_depth_xgb', 2, 20)
            model = XGBClassifier(
                n_estimators=n_estimators_xgb,
                learning_rate=learning_rate_xgb,
                max_depth=max_depth_xgb,
                random_state=42,
                n_jobs=-1,
                verbosity=0,
            )

        elif model_name == 'LGBM':
            n_estimators_lgbm  = trial.suggest_int('n_estimators_lgbm', 10, 200)
            learning_rate_lgbm = trial.suggest_float('learning_rate_lgbm', 0.01, 0.5)
            max_depth_lgbm     = trial.suggest_int('max_depth_lgbm', 2, 20)
            model = LGBMClassifier(
                n_estimators=n_estimators_lgbm,
                learning_rate=learning_rate_lgbm,
                max_depth=max_depth_lgbm,
                random_state=42,
                n_jobs=-1,
                verbose=-1,
            )

        elif model_name == 'SVM':
            C_svm = trial.suggest_float('C_svm', 0.1, 10.0)
            model = SVC(C=C_svm, probability=True, random_state=42)

        # train on resampled data
        model.fit(X_train_resampled, y_train_resampled)

        # log model params
        mlflow.log_params(model.get_params())

        # predictions on held-out test set (full, unresampled)
        y_pred_prob = model.predict_proba(X_test_trans)[:, 1]

        # primary metric
        score = roc_auc_score(y_test, y_pred_prob)

        # log model name + metric
        mlflow.log_param('model', model_name)
        mlflow.log_metric('ROC_AUC', score)

        return score


In [ ]:
# create optuna study
study = optuna.create_study(direction='maximize', study_name='model_selection')

with mlflow.start_run(run_name='Best Model') as parent:
    # optimize the objective function
    study.optimize(objective, n_trials=30, n_jobs=-1)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric('best_score', study.best_value)


In [ ]:
# best score
study.best_value


In [ ]:
# best parameters
study.best_params


In [ ]:
# dataframe of all trials
study.trials_dataframe()


In [ ]:
# model frequency across all trials
study.trials_dataframe()['params_model'].value_counts()


In [ ]:
# avg scores for all tested models
avg_scores = (
    study.trials_dataframe()
    .groupby('params_model')['value']
    .mean()
    .sort_values(ascending=False)
)
print('=== Average ROC-AUC per Model Family ===')
print(avg_scores)


## Optimisation History

In [ ]:
# optimization history plot
optuna.visualization.plot_optimization_history(study)


In [ ]:
# partial coord plot -- model family comparison
optuna.visualization.plot_parallel_coordinate(study, params=['model'])


## Evaluate Best Model on Test Set

In [ ]:
# Identify winning model family from study
best_model_name = study.best_params['model']
print('Best model family:', best_model_name)
print('Best ROC-AUC     :', round(study.best_value, 4))


In [ ]:
# Rebuild best model using best params from study
best_params = study.best_params.copy()
best_params.pop('model')   # remove the 'model' key — remaining are hyperparams

if best_model_name == 'LR':
    best_model = LogisticRegression(C=best_params.get('C_lr', 1.0), max_iter=500, random_state=42)

elif best_model_name == 'DT':
    best_model = DecisionTreeClassifier(max_depth=best_params.get('max_depth_dt', 10), random_state=42)

elif best_model_name == 'KNN':
    best_model = KNeighborsClassifier(n_neighbors=best_params.get('n_neighbors_knn', 5), n_jobs=-1)

elif best_model_name == 'RF':
    best_model = RandomForestClassifier(
        n_estimators=best_params.get('n_estimators_rf', 100),
        max_depth=best_params.get('max_depth_rf', 10),
        random_state=42, n_jobs=-1,
    )

elif best_model_name == 'XGB':
    best_model = XGBClassifier(
        n_estimators=best_params.get('n_estimators_xgb', 100),
        learning_rate=best_params.get('learning_rate_xgb', 0.1),
        max_depth=best_params.get('max_depth_xgb', 6),
        random_state=42, n_jobs=-1, verbosity=0,
    )

elif best_model_name == 'LGBM':
    best_model = LGBMClassifier(
        n_estimators=best_params.get('n_estimators_lgbm', 100),
        learning_rate=best_params.get('learning_rate_lgbm', 0.1),
        max_depth=best_params.get('max_depth_lgbm', 6),
        random_state=42, n_jobs=-1, verbose=-1,
    )

elif best_model_name == 'SVM':
    best_model = SVC(C=best_params.get('C_svm', 1.0), probability=True, random_state=42)

# train on resampled data
best_model.fit(X_train_resampled, y_train_resampled)
print('Best model trained.')


In [ ]:
# Predictions on full train + test (pre-transformed, unresampled)
y_pred_train = best_model.predict(X_train_trans)
y_pred_test  = best_model.predict(X_test_trans)
y_prob_train = best_model.predict_proba(X_train_trans)[:, 1]
y_prob_test  = best_model.predict_proba(X_test_trans)[:, 1]

print(f'Train accuracy : {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Test  accuracy : {accuracy_score(y_test, y_pred_test):.4f}')
print(f'Train recall   : {recall_score(y_train, y_pred_train):.4f}')
print(f'Test  recall   : {recall_score(y_test, y_pred_test):.4f}')
print(f'Train ROC-AUC  : {roc_auc_score(y_train, y_prob_train):.4f}')
print(f'Test  ROC-AUC  : {roc_auc_score(y_test, y_prob_test):.4f}')


In [ ]:
print('\n=== Classification Report (Test Set) ===')
print(classification_report(y_test, y_pred_test, target_names=['Stayed', 'Churned']))


In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Churned'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — Best Model: {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.show()


## Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'roc_auc'  : 'roc_auc',
    'recall'   : 'recall',
    'precision': 'precision',
    'f1'       : 'f1',
    'accuracy' : 'accuracy',
}


In [ ]:
# Full pipeline: preprocessor + KNN imputer + resampler + best classifier
cv_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn_imputer',  KNNImputer(n_neighbors=10)),
    ('sampler',      RandomUnderSampler(random_state=42)),
    ('classifier',   best_model),
])

cv_results = cross_validate(
    cv_pipeline,
    X_train,       # raw X_train — pipeline handles everything inside each fold
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

results_df = pd.DataFrame(cv_results)
results_df


In [ ]:
print('=== CV Summary (5-Fold Stratified) ===')
print(f"Mean CV ROC-AUC  : {results_df['test_roc_auc'].mean():.4f} +/- {results_df['test_roc_auc'].std():.4f}")
print(f"Mean CV Recall   : {results_df['test_recall'].mean():.4f} +/- {results_df['test_recall'].std():.4f}")
print(f"Mean CV Precision: {results_df['test_precision'].mean():.4f} +/- {results_df['test_precision'].std():.4f}")
print(f"Mean CV F1       : {results_df['test_f1'].mean():.4f} +/- {results_df['test_f1'].std():.4f}")
print(f"Mean CV Accuracy : {results_df['test_accuracy'].mean():.4f} +/- {results_df['test_accuracy'].std():.4f}")


## Log Experiment to MLflow

In [ ]:
try:
    with mlflow.start_run(run_name=f'Best Model — {best_model_name}'):
        mlflow.log_param('experiment_type',    'Model Selection')
        mlflow.log_param('imbalance_strategy', 'RandomUnderSampler')
        mlflow.log_param('imputer',            'KNNImputer(n_neighbors=10)')
        mlflow.log_param('best_model',         best_model_name)
        mlflow.log_param('dataset_rows',       X.shape[0])
        mlflow.log_param('dataset_features',   X.shape[1])
        mlflow.log_params(best_model.get_params())

        mlflow.log_metric('training_accuracy', accuracy_score(y_train, y_pred_train))
        mlflow.log_metric('training_recall',   recall_score(y_train, y_pred_train))
        mlflow.log_metric('training_roc_auc',  roc_auc_score(y_train, y_prob_train))
        mlflow.log_metric('test_accuracy',     accuracy_score(y_test, y_pred_test))
        mlflow.log_metric('test_recall',       recall_score(y_test, y_pred_test))
        mlflow.log_metric('test_precision',    precision_score(y_test, y_pred_test))
        mlflow.log_metric('test_f1',           f1_score(y_test, y_pred_test))
        mlflow.log_metric('test_roc_auc',      roc_auc_score(y_test, y_prob_test))
        mlflow.log_metric('cv_mean_recall',    results_df['test_recall'].mean())
        mlflow.log_metric('cv_mean_roc_auc',   results_df['test_roc_auc'].mean())
        mlflow.log_metric('cv_mean_f1',        results_df['test_f1'].mean())

        for i, s in enumerate(results_df['test_roc_auc']):
            mlflow.log_metric('cv_roc_auc_fold', s, step=i)
        for i, s in enumerate(results_df['test_recall']):
            mlflow.log_metric('cv_recall_fold', s, step=i)

    print('Experiment logged to MLflow.')
except Exception as e:
    print(f'[MLflow Notice] {e}')


## Final Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Test ROC-AUC', 'Test Recall', 'Test Precision', 'Test F1', 'Test Accuracy',
        'CV ROC-AUC (mean)', 'CV Recall (mean)', 'CV F1 (mean)', 'CV Accuracy (mean)',
    ],
    f'Exp 4 — Model Selection ({best_model_name} + UnderSample)': [
        round(roc_auc_score(y_test, y_prob_test),           4),
        round(recall_score(y_test, y_pred_test),            4),
        round(precision_score(y_test, y_pred_test),         4),
        round(f1_score(y_test, y_pred_test),                4),
        round(accuracy_score(y_test, y_pred_test),          4),
        round(results_df['test_roc_auc'].mean(),            4),
        round(results_df['test_recall'].mean(),             4),
        round(results_df['test_f1'].mean(),                 4),
        round(results_df['test_accuracy'].mean(),           4),
    ]
})
summary


In [ ]:
print(f'>>> Best model: {best_model_name}  |  Optuna best ROC-AUC: {study.best_value:.4f}')
print('>>> Average ROC-AUC per model family (ranked):')
print(avg_scores.to_string())
print()
print('>>> Ready for dedicated hyperparameter tuning in the next notebook.')
